# ADS: Attention Divergence Score — Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/djokobandjur/ads-vit-forensics/blob/main/colab_quickstart.ipynb)

This notebook reproduces the full pipeline for the paper *"Attention Divergence Score: A Forensic Metric for Characterizing Parameter-Level Attacks in Vision Transformers"* (resubmission).

**Layout:**
- Repository scripts are cloned fresh into `/content/ads-vit-forensics/`.
- Trained models live on Google Drive (reused from the [companion work](https://github.com/djokobandjur/vit-positional-adversarial); large, persistent).
- ImageNet-100 val/ directory comes from the companion work's `00_setup_imagenet.py` utility.
- CIFAR-100 is auto-downloaded by torchvision into `/content/drive/MyDrive/cifar100_cache/`.
- All generated JSON results are written to `/content/drive/MyDrive/ads_pipeline/data/`.
- Figures are written to `/content/drive/MyDrive/ads_pipeline/output/figures/`.

**Hardware:** The experiments in this work were run on Google Cloud G4 VMs equipped with NVIDIA RTX PRO 6000 Blackwell Server Edition GPUs (96 GB GDDR7 VRAM, CUDA 13.0). The verification step (Section 10) runs in seconds on CPU without GPU or PyTorch. Each section can be re-run independently provided prior JSON outputs are present on Drive.

**Per-buffer attack convention:** This work uses a per-buffer independent δ_i for each replicated PE buffer (12 per block for RoPE, 12 for ALiBi), differing from the shared-δ convention used in the companion work. The choice is methodologically motivated; see Section II.E of the manuscript and Table II for side-by-side comparison.

## 1. Setup

Mount Google Drive (for trained models, datasets, and persistent JSON output), verify GPU availability, and clone the repository into `/content/`.

### 1.1 Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 1.2 Verify GPU

All experiment scripts require a CUDA-capable GPU. The pipeline was developed on Google Cloud G4 VMs with NVIDIA RTX PRO 6000 Blackwell GPUs (96 GB VRAM, CUDA 13.0). The verification step (Section 10) does not require a GPU.

In [2]:
!nvidia-smi

Mon May 18 06:17:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   27C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### 1.3 Clone repositories

This work uses `VisionTransformer` (the ViT model class with 4 PE implementations) from the companion work [vit-positional-adversarial](https://github.com/djokobandjur/vit-positional-adversarial). All 11 ADS experiment scripts import it via `from full_scale_experiment import VisionTransformer`, expecting the file at `/content/full_scale_experiment.py`.

Both repositories are cloned in this step; `full_scale_experiment.py` is then copied to `/content/` so the ADS scripts can find it.

In [2]:
%cd /content
!git clone https://github.com/djokobandjur/ads-vit-forensics.git
!git clone https://github.com/djokobandjur/vit-positional-adversarial.git

# ADS scripts import VisionTransformer via `from full_scale_experiment import ...`
# with sys.path.insert(0, '/content'), so we copy the companion file there.
!cp /content/vit-positional-adversarial/full_scale_experiment.py /content/

%cd /content/ads-vit-forensics

/content
Cloning into 'ads-vit-forensics'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 105 (delta 43), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 1.53 MiB | 14.79 MiB/s, done.
Resolving deltas: 100% (43/43), done.
Cloning into 'vit-positional-adversarial'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 403 (delta 56), reused 3 (delta 3), pack-reused 287 (from 2)
Receiving objects: 100% (403/403), 974.24 KiB | 10.71 MiB/s, done.
Resolving deltas: 100% (220/220), done.
/content/ads-vit-forensics


### 1.4 Create output directories on Drive

In [4]:
import os

DATA_DIR = '/content/drive/MyDrive/ads_pipeline/data'
FIG_DIR  = '/content/drive/MyDrive/ads_pipeline/output/figures'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('Output dirs ready:')
print(' ', DATA_DIR)
print(' ', FIG_DIR)

Output dirs ready:
  /content/drive/MyDrive/ads_pipeline/data
  /content/drive/MyDrive/ads_pipeline/output/figures


## 2. Dataset Preparation

ImageNet-100 requires the companion work's setup utility to extract the 100 selected classes from the ILSVRC2012 val tarball. CIFAR-100 is downloaded automatically by torchvision.

### 2.1 ImageNet-100 setup (via companion work)

Run the companion repository's `00_setup_imagenet.py` utility, which unpacks 100 classes (50 images per class) from the ILSVRC2012 val tar archive on Drive into `/content/imagenet100/val/`. The companion repository was cloned in Section 1.3.

Expected input: `/content/drive/MyDrive/pe_experiment/imagenet/ILSVRC2012_img_val.tar`

In [3]:
# Run the companion repo's ImageNet-100 setup utility
# (companion repo was already cloned in Section 1.3)
!python /content/drive/MyDrive/pe_experiment/ads_scripts/00_setup_imagenet.py


ImageNet-100 Setup
 Found: ILSVRC2012_img_val.tar
 Found: imagenet100_classes.txt
 Found: val_labels.txt at /content/drive/My Drive/pe_experiment/val_labels.txt

ImageNet-100 classes: 100
Val labels loaded: 50000 entries
Output directory: /content/imagenet100/val
Created 100 class folders

Extracting from: /content/drive/My Drive/pe_experiment/imagenet/ILSVRC2012_img_val.tar
(This may take 5-10 minutes depending on Drive speed...)

Total images in tar: 50,000
Filtering: 100% 50000/50000 [00:02<00:00, 16923.42img/s]

Extraction complete!
  Images extracted: 5,000
  Images skipped:   45,000
  Expected:         5,000

 All 5,000 images extracted successfully.

Per-class image count (first 5):
  n01558993: 50 images
  n01692333: 50 images
  n01729322: 50 images
  n01735189: 50 images
  n01749939: 50 images

  Min images per class: 50
  Max images per class: 50
All classes have exactly 50 images.

Dataset ready at: /content/imagenet100/val


### 2.2 CIFAR-100

No preparation needed: the scripts call `torchvision.datasets.CIFAR100(download=True)` internally with cache at `/content/drive/MyDrive/cifar100_cache/`.

## 3. Main ADS Experiments

PE-only PGD attack on all 24 models (4 PE × 3 seeds × 2 datasets) with ADS computation at every ε. Produces the per-layer ADS heatmaps and ADS(L4) trajectories that drive Sections 5 and 6 of the manuscript.

Output: nested JSON `{pe_type: {seed: {accuracies, ads_layer4, ads_per_layer, ...}}}` for each ε in `[0.0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0]`.

### 3.1 ImageNet-100

In [11]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_experiment.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_results.json" \
     --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
Loaded reference indices from /content/drive/MyDrive/ads_pipeline/data/ads_ref_indices.json
Reference images: 256
Validation images: 5000
Attention Divergence Score (ADS) Experiment — ImageNet-100
Attack: PGD (T=20 steps)
PE types: ['learned', 'sinusoidal', 'rope', 'alibi']
Seeds: [42]
Epsilons: [0.0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0]
Reference images: 256
Results: /content/drive/MyDrive/ads_pipeline/data/ads_results.json


PE TYPE: LEARNED

  Seed 42:
    Model loaded
    Computing clean attention baseline (256 images)...
    Clean accuracy: 79.44%
    ε=0.000: acc=79.4%, ADS(mean)=0.0000, ADS(mid)=0.0000, ADS(L4)=0.0000
    ε=0.001: acc=79.4%, ADS(mean)=0.0000, ADS(mid)=0.0000, ADS(L4)=0.0000
    ε=0.005: acc=79.3%, ADS(mean)=0.0001, ADS(mid)=0.0000, ADS(L4)=0.0002
    ε=0.010: acc=79.2%, ADS(mean)=0.0003, ADS(mid)=0.0001, ADS(L4)=0.0010
    ε=0.050: acc=78.6%, ADS(mean)=0.0072, ADS(mid)=0.0034, ADS(L4)=0.0241
    ε=0.10

### 3.2 CIFAR-100

In [12]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_experiment_cifar.py \
    --models_dir "/content/drive/MyDrive/Trained models_CIFAR100" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_results_cifar100.json" \
    --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
100% 169M/169M [00:04<00:00, 39.2MB/s]
Loaded reference indices from /content/drive/MyDrive/ads_pipeline/data/ads_ref_indices.json
Reference images: 256
Validation images: 10000
Attention Divergence Score (ADS) Experiment — CIFAR-100
Attack: PGD (T=20 steps)
PE types: ['learned', 'sinusoidal', 'rope', 'alibi']
Seeds: [42]
Epsilons: [0.0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0]
Reference images: 256
Results: /content/drive/MyDrive/ads_pipeline/data/ads_results_cifar100.json


PE TYPE: LEARNED

  Seed 42:
    Model loaded
    Computing clean attention baseline (256 images)...
    Clean accuracy: 68.72%
    ε=0.000: acc=68.7%, ADS(mean)=0.0000, ADS(mid)=0.0000, ADS(L4)=0.0000
    ε=0.001: acc=68.8%, ADS(mean)=0.0000, ADS(mid)=0.0000, ADS(L4)=0.0000
    ε=0.005: acc=68.3%, ADS(mean)=0.0007, ADS(mid)=0.0003, ADS(L4)=0.0006
    ε=0.010: acc=67.9%, ADS(mean)=0.0028, ADS(mid)=0.0011, ADS(L4)=0.0023
    ε=0.050: acc=46.7%, ADS(mean)=0.04

## 4. Attack-Surface Specificity

Compares four attack surfaces (PE-only, QKV-only, MLP-only, all-weights) at matched per-buffer perturbation granularity. Produces the saturation-budget asymmetry analysis (Section 6.2): PE-only attacks require 17×–200× larger budgets than weight attacks for operational compromise.

### 4.1 ImageNet-100

In [13]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_specificity.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_specificity.json" \
    --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
Loaded reference indices from /content/drive/MyDrive/ads_pipeline/data/ads_ref_indices.json
Reference: 256 images, Val: 5000 images
ADS Specificity Experiment
PE types: ['learned', 'rope', 'sinusoidal', 'alibi']
Seeds: [42]
Perturbation types: ['pe_only', 'qkv_only', 'mlp_only', 'all_weights']
Epsilons: [0.0, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]


PE TYPE: LEARNED

  Seed 42:
    Clean accuracy: 79.44%

    Perturbation: PE_ONLY
      ε=0.000: acc=79.4%, ADS(mean)=0.0000, ADS(mid)=0.0000, ADS(L4)=0.0000
      ε=0.005: acc=79.3%, ADS(mean)=0.0001, ADS(mid)=0.0000, ADS(L4)=0.0002
      ε=0.010: acc=79.3%, ADS(mean)=0.0003, ADS(mid)=0.0001, ADS(L4)=0.0010
      ε=0.050: acc=78.5%, ADS(mean)=0.0072, ADS(mid)=0.0034, ADS(L4)=0.0243
      ε=0.100: acc=76.4%, ADS(mean)=0.0289, ADS(mid)=0.0119, ADS(L4)=0.0829
      ε=0.200: acc=56.8%, ADS(mean)=0.0859, ADS(mid)=0.0399, ADS(L4)=0.2511
      ε=0.500: acc=3.6%, ADS(mean)=0.2600, ADS(mid)=0.1706, ADS(L4)=

### 4.2 CIFAR-100

In [14]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_specificity_cifar.py \
    --models_dir "/content/drive/MyDrive/Trained models_CIFAR100" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_specificity_cifar.json" \
    --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
Loaded reference indices from /content/drive/MyDrive/ads_pipeline/data/ads_ref_indices.json
Reference: 256 images, Val: 10000 images
ADS Specificity Experiment
PE types: ['learned', 'rope', 'sinusoidal', 'alibi']
Seeds: [42]
Perturbation types: ['pe_only', 'qkv_only', 'mlp_only', 'all_weights']
Epsilons: [0.0, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]


PE TYPE: LEARNED

  Seed 42:
    Clean accuracy: 68.72%

    Perturbation: PE_ONLY
      ε=0.000: acc=68.7%, ADS(mean)=0.0000, ADS(mid)=0.0000, ADS(L4)=0.0000
      ε=0.005: acc=68.3%, ADS(mean)=0.0007, ADS(mid)=0.0003, ADS(L4)=0.0006
      ε=0.010: acc=67.8%, ADS(mean)=0.0028, ADS(mid)=0.0011, ADS(L4)=0.0024
      ε=0.050: acc=46.4%, ADS(mean)=0.0486, ADS(mid)=0.0189, ADS(L4)=0.0534
      ε=0.100: acc=13.3%, ADS(mean)=0.1204, ADS(mid)=0.0484, ADS(L4)=0.1418
      ε=0.200: acc=3.1%, ADS(mean)=0.2234, ADS(mid)=0.0880, ADS(L4)=0.2586
      ε=0.500: acc=1.5%, ADS(mean)=0.3095, ADS(mid)=0.1283, ADS(L4)=

## 5. Adaptive Attacker (Section 6.7)

Tests whether an attacker who knows the ADS metric can evade detection by adding an ADS-minimization regularization term to the PGD loss: `L_evasion = -L_CE + λ · ADS(δ)`. Confirms that ADS is a structural consequence of perturbation magnitude — no setting of λ yields evasion with damage. Verified with SPSA (gradient-free zeroth-order attacker) as required by Carlini et al. (2019).

In [15]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_adaptive.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_adaptive.json" \
    --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
Reference: 256 images, Val: 5000 images
Adaptive Attacker Experiment
PE types: ['learned', 'rope'], Seeds: [42]
Epsilons: [0.05, 0.1, 0.2], Lambdas: [0.0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0]
PGD steps: 20
Detection thresholds: {'learned': 2.65, 'rope': 5.01}


PE TYPE: LEARNED
Detection threshold: 2.65x baseline

  Seed 42:
    Clean accuracy: 79.44%

    ε=0.05:
      λ=  0.0: acc_drop=  0.18pp, ADS=0.023887 (1.54x), EVADES ✅
      λ=  0.1: acc_drop=  0.28pp, ADS=0.023818 (1.54x), EVADES ✅
      λ=  0.5: acc_drop=  0.24pp, ADS=0.022965 (1.48x), EVADES ✅
      λ=  1.0: acc_drop= -0.02pp, ADS=0.023743 (1.53x), EVADES ✅
      λ=  2.0: acc_drop=  0.10pp, ADS=0.022844 (1.47x), EVADES ✅
      λ=  5.0: acc_drop=  0.38pp, ADS=0.023866 (1.54x), EVADES ✅
      λ= 10.0: acc_drop=  0.08pp, ADS=0.024017 (1.55x), EVADES ✅
      λ= 50.0: acc_drop=  0.20pp, ADS=0.024149 (1.56x), EVADES ✅

    ε=0.1:
      λ=  0.0: acc_drop=  1.48pp, ADS=0.087639 (5.65x), DETE

## 6. Threshold Universality and Reference Evasion

### 6.1 Fine ε grid + interpolated threshold

Tests universality of the detection threshold (10× baseline ADS) with a finer ε grid (14 values from 0.001 to 1.0) and log-log interpolation of the actual crossing point. Statistical test (Kruskal-Wallis) confirms whether thresholds differ significantly across PE types.

In [16]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_threshold_fine.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_threshold_fine.json" \
    --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
Loaded 256 reference indices
ADS Fine-Grid Threshold Experiment
PE types: ['learned', 'sinusoidal', 'rope', 'alibi']
Seeds: [42]
Fine-grid epsilons: [0.0, 0.001, 0.0015, 0.002, 0.003, 0.005, 0.007, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]


PE TYPE: LEARNED

  Seed 42:
    ε=0.0000: ADS(L4)=0.000000
    ε=0.0010: ADS(L4)=0.000010
    ε=0.0015: ADS(L4)=0.000022
    ε=0.0020: ADS(L4)=0.000037
    ε=0.0030: ADS(L4)=0.000086
    ε=0.0050: ADS(L4)=0.000229
    ε=0.0070: ADS(L4)=0.000473
    ε=0.0100: ADS(L4)=0.000940
    ε=0.0200: ADS(L4)=0.003574
    ε=0.0500: ADS(L4)=0.023400
    ε=0.1000: ADS(L4)=0.085111
    ε=0.2000: ADS(L4)=0.259238
    ε=0.5000: ADS(L4)=0.726343
    ε=1.0000: ADS(L4)=1.046323
    Baseline ADS(L4) at ε=0.001: 0.000010
    Interpolated 10x threshold: ε = 0.00320

  Saved to /content/drive/MyDrive/ads_pipeline/data/ads_threshold_fine.json

PE TYPE: SINUSOIDAL

  Seed 42:
    ε=0.0000: ADS(L4)=0.000000
    ε=0.0010: ADS(L4)=0.000002


### 6.2 Reference set evasion

Tests whether an attacker who knows the 256 reference image indices can craft a perturbation that preserves attention on the reference set (evades ADS) while degrading accuracy on the full validation set. Probes whether reference set secrecy is required for ADS detection.

In [17]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_ref_evasion.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_ref_evasion.json" \
    --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
Reference set: 256 images (known to attacker)
Holdout set:   256 images (unseen by attacker)
Reference Set Evasion Experiment
PE types: ['learned', 'rope'], Seeds: [42]
Epsilons: [0.1, 0.2, 0.5], Lambdas: [0.0, 1.0, 5.0, 10.0, 50.0]
Reference: 256 known images, Holdout: 256 unseen images


PE TYPE: LEARNED

  Seed 42:

    ε=0.1:
         λ    Acc drop    ADS(ref)   ADS(hold)    Evasion?   Generalized?
         0        1.24pp      5.54x       5.62x          NO             NO
         1        1.22pp      5.76x       5.84x          NO             NO
         5        1.34pp      5.58x       5.65x          NO             NO
        10        1.30pp      5.21x       5.28x          NO             NO
        50        1.40pp      5.78x       5.85x          NO             NO

    ε=0.2:
         λ    Acc drop    ADS(ref)   ADS(hold)    Evasion?   Generalized?
         0        9.26pp     17.56x      17.75x          NO             NO
         1       

## 7. Residual Stream Probing

Layer-wise positional probing: for each Transformer block, trains a Ridge regression probe to predict patch position (row, col) from 768-d residual stream activations. Uses image-level GroupKFold cross-validation (5 folds) to prevent data leakage from correlated patch activations.

The peak probing layer per PE type provides one axis of the three-way convergence finding (Section 7), validating that the operation-space partition (embedding-space vs. attention-space) is consistent across attention metrics, residual-stream metrics, and ADS signatures.

### 7.1 ImageNet-100

In [18]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_probing_residual.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_probing_residual.json" \
    --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
Grid: 14×14 = 196 patches
Embed dim: 768, Feature per patch: 768
Reference set: 256 images
Layer-wise Positional Probing — Residual Stream
PE types: ['learned', 'sinusoidal', 'rope', 'alibi'], Seeds: [42]
Feature: residual stream patch activations, dim=768
CV: image-level GroupKFold, 5 folds
Samples: 256 images × 196 patches = 50176


PE TYPE: LEARNED

  Seed 42:
    Extracting residual stream activations...
    Samples: 50176 (256 images × 196 patches)
    Feature dim: 768

      Layer     R²(row)     R²(col)    R²(mean)
    ---------------------------------------------
    Layer  1:      0.9884      0.7502      0.8693
    Layer  2:      0.9848      0.4244      0.7046
    Layer  3:      0.9817      0.2389      0.6103
    Layer  4:      0.9768      0.1203      0.5485 ← L4
    Layer  5:      0.9653      0.0565      0.5109
    Layer  6:      0.9468      0.0203      0.4836
    Layer  7:      0.9353     -0.0022      0.4666
    Layer  8:      0.9275 

### 7.2 CIFAR-100

In [19]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_probing_residual_cifar.py \
    --models_dir "/content/drive/MyDrive/Trained models_CIFAR100" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_probing_residual_cifar.json" \
    --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
Grid: 8×8 = 64 patches
Embed dim: 768, Feature per patch: 768
Reference set: 256 images
Layer-wise Positional Probing — Residual Stream
PE types: ['learned', 'sinusoidal', 'rope', 'alibi'], Seeds: [42]
Feature: residual stream patch activations, dim=768
CV: image-level GroupKFold, 5 folds
Samples: 256 images × 64 patches = 16384


PE TYPE: LEARNED

  Seed 42:
    Extracting residual stream activations...
    Samples: 16384 (256 images × 64 patches)
    Feature dim: 768

      Layer     R²(row)     R²(col)    R²(mean)
    ---------------------------------------------
    Layer  1:      0.9915      0.9824      0.9870
    Layer  2:      0.9900      0.9558      0.9729
    Layer  3:      0.9883      0.9211      0.9547
    Layer  4:      0.9857      0.8916      0.9386 ← L4
    Layer  5:      0.9833      0.8706      0.9270
    Layer  6:      0.9816      0.8516      0.9166
    Layer  7:      0.9795      0.8353      0.9074
    Layer  8:      0.9780      

## 8. ROC Analysis vs Benign Shifts

Tests the operational detection boundary: ROC AUC of ADS as a binary detector between adversarial PGD-PE perturbations and realistic benign image shifts (JPEG compression, Gaussian blur, additive noise). Confirms AUC ≥ 0.82 at ε ≥ 0.1 (Learned) / ε ≥ 0.2 (RoPE), establishing operational detectability under domain shift.

In [20]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_roc_analysis.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_roc_v2.json" \
    --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
Reference set: 256 images
ADS ROC Analysis v2 — Corrected Design
PE types: ['learned', 'rope'], Seeds: [42]
N_BOOTSTRAP: 20 splits for clean noise floor


PE TYPE: LEARNED

  Seed 42:
    Reference attention computed (256 images)
    Computing clean noise floor (20 bootstrap splits)...
    Clean noise: mean=0.017639, std=0.000508, CV=2.9%
    Computing benign distribution shifts...
      jpeg_q50    : 0.000761 (0.0x baseline)
      jpeg_q30    : 0.001650 (0.1x baseline)
      jpeg_q10    : 0.006563 (0.4x baseline)
      blur_s1     : 0.005209 (0.3x baseline)
      blur_s2     : 0.024520 (1.4x baseline)
      blur_s3     : 0.034174 (1.9x baseline)
      noise_005   : 0.000076 (0.0x baseline)
      noise_010   : 0.000396 (0.0x baseline)
      noise_020   : 0.001567 (0.1x baseline)
    Computing PE attack scores...
      ε=0.001: 0.000010 (0.0x baseline)
      ε=0.002: 0.000040 (0.0x baseline)
      ε=0.003: 0.000090 (0.0x baseline)
      ε=0.005: 

## 9. PE-to-PE Comparison and Figure Generation

### 9.1 PE-to-PE comparison

Auxiliary analysis comparing ADS signatures between PE pairs (e.g. RoPE vs ALiBi attention-space behavior).

In [21]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/ads_comparison.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_comparison.json" \
    --seeds 42

Device: cuda
[CLI override] SEEDS = [42]
Detection Method Comparison Experiment
PE types: ['learned', 'rope'], Seeds: [42]
Methods: ADS(L4), Mahalanobis(attention), LogitKL


PE TYPE: LEARNED

  Seed 42:
    Computing clean noise floors...
    Clean baselines — ADS: 0.017639, Mah: 0.0000, LogitKL: 0.599919
    Evaluating attacks...
      ε=0.001: ADS AUC=0.00, Mah AUC=1.00, LogitKL AUC=0.00
      ε=0.005: ADS AUC=0.00, Mah AUC=1.00, LogitKL AUC=0.00
      ε=0.010: ADS AUC=0.00, Mah AUC=1.00, LogitKL AUC=0.00
      ε=0.050: ADS AUC=1.00, Mah AUC=1.00, LogitKL AUC=0.00
      ε=0.100: ADS AUC=1.00, Mah AUC=1.00, LogitKL AUC=0.00
      ε=0.200: ADS AUC=1.00, Mah AUC=1.00, LogitKL AUC=0.00
      ε=0.500: ADS AUC=1.00, Mah AUC=1.00, LogitKL AUC=0.00

  Saved to /content/drive/MyDrive/ads_pipeline/data/ads_comparison.json

PE TYPE: ROPE

  Seed 42:
    Computing clean noise floors...
    Clean baselines — ADS: 0.013270, Mah: 0.0000, LogitKL: 0.643493
    Evaluating attacks...
      ε=0.001: A

### 9.2 Generate paper figures

Produces 4 figures (matching manuscript v20 numbering) from the JSON files.

In [5]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/generate_ads_figures.py \
    --data-dir   "/content/drive/MyDrive/ads_pipeline/data_smoke_tests" \
  #  --output-dir "/content/drive/MyDrive/ads_pipeline/output"


Loaded data from /content/drive/MyDrive/ads_pipeline/data_smoke_tests/
Saving figures to /content/output/figures/

Seeds detected (all PE×datasets): 42
⚠  Fewer than 3 seeds present (min: 1). Figures will reflect available seeds; std bands and slopes are NOT publication statistics with n<3.

✓ Fig 1 saved: ads_fig1_l4_vs_epsilon.png
✓ Fig 2 saved: ads_fig2_per_layer_heatmap.png
✓ Fig 3 saved: ads_fig3_layer_profile.png
✓ Fig 4 saved: ads_fig4_early_warning_combined.png

✅ All 4 ADS figures generated in /content/output/figures/


## 10. CPU-Only Verification (~30 seconds)

`reproduce.py` reads only the archived JSON files and regenerates every numerical claim, table value, and statistical test in the manuscript with PASS/FAIL tolerances. **Requires no GPU and no PyTorch** — pure numpy/scipy/pandas. This is the canonical verification step for reviewers and external readers.

Set `--data-dir` to point at the JSON output directory from earlier sections. The script writes verification tables to `output/tables/`.

In [20]:
# Copy JSONs from Drive to local data/ so reproduce.py finds them
import shutil
from pathlib import Path

local_data = Path('/content/ads-vit-forensics/data')
local_data.mkdir(exist_ok=True)

drive_data = Path('/content/drive/MyDrive/ads_pipeline/data')
for f in drive_data.glob('*.json'):
    shutil.copy(f, local_data / f.name)

print(f'Copied {len(list(local_data.glob("*.json")))} JSON files to {local_data}')

Copied 12 JSON files to /content/ads-vit-forensics/data


In [7]:
!python /content/drive/MyDrive/pe_experiment/ads_scripts/reproduce.py \
--data-dir   "/content/drive/MyDrive/ads_pipeline/data_smoke test seed 42"



REPRODUCTION RUN — 2026-05-18T13:55:40.426027
Paper: 'Attention Divergence Score: A Forensic Metric for
        Characterizing Parameter-Level Attacks in Vision Transformers'
Submitted to: IEEE TIFS
Data dir:     /content/drive/MyDrive/ads_pipeline/data_smoke test seed 42
Output dir:   output

STEP 1: Loading data
  [OK]   Loaded ads_results.json
  [OK]   Loaded ads_results_cifar100.json
  [OK]   Loaded ads_specificity.json
  [OK]   Loaded ads_specificity_cifar.json
  [OK]   Loaded ads_probing_residual.json
  [OK]   Loaded ads_probing_residual_cifar.json
  [OK]   Loaded ads_threshold_fine.json
  [OK]   Loaded ads_roc_v2.json
  [OK]   Loaded ads_comparison.json
  [OK]   Loaded ads_ref_evasion.json
  [OK]   Loaded ads_adaptive.json
  [SKIP] ads_ref_indices.json not found at /content/drive/MyDrive/ads_pipeline/data_smoke test seed 42/ads_ref_indices.json

ABORT: reproduce.py expects all 3 seeds (42, 123, 456) per PE type.
This script reproduces *paper* numbers (CIs, ANOVA, ICC, 1-NN LOO)

## Done

All experimental artifacts are now on Drive:

- **13 JSON result files** in `/content/drive/MyDrive/ads_pipeline/data/`
- **Paper figures** in `/content/drive/MyDrive/ads_pipeline/output/figures/`
- **Verification tables** in `/content/ads-vit-forensics/output/tables/`

For methodological details on the per-buffer attack convention used in this work (and its relationship to the shared-δ convention in the companion work), see Section II.E and Table II of the manuscript, and `CHANGELOG.md` in this repository.